In [87]:
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

In [88]:
from langchain_community.utilities import SQLDatabase

In [89]:
host = 'localhost'                                                                                                                  
port = '3306'
username = 'root'
password = ''
database_schema = 'text_to_sql'
mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"

In [90]:
db = SQLDatabase.from_uri(mysql_uri)

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")

Dialect: mysql
Available tables: ['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'state_regions']


In [91]:
import sys
import subprocess

# Install the pymysql driver directly into your kernel
subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-google-genai"])


0

In [101]:

from dotenv import load_dotenv
import os
from dotenv import load_dotenv
load_dotenv()  # loads GOOGLE_API_KEY from .env file

from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [102]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [103]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002466569E3F0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002466569E3F0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002466569E3F0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [105]:
from langchain_community.tools.sql_database.tool import (
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
    QuerySQLCheckerTool,
    QuerySQLDatabaseTool,
)

In [106]:
import sys
import subprocess

# Install the pymysql driver directly into your kernel
subprocess.check_call([sys.executable, "-m", "pip", "install", "langchainhub"])



0

In [107]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an agent designed to interact with a SQL database.\n"
               "Given an input question, create a syntactically correct {dialect} query to run, "
               "then look at the results of the query and return the answer.\n"
               "Always limit your query to at most {top_k} results unless the user specifies otherwise.\n"
               "You can order the results by a relevant column to return the most interesting data.\n"
               "Never query for all the columns from a specific table, only ask for the relevant columns given the question.\n"
               "You have access to tools for interacting with the database.\n"
               "Only use the below tools. Only use the information returned by the below tools to construct your final answer.\n"
               "You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\n"
               "DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.\n\n"
               "To start you should ALWAYS look at the tables in the database to see what you can query.\n"
               "Do NOT skip this step."),
    ("placeholder", "{messages}"),
])

In [108]:
system_message = prompt_template.format(dialect="mysql", top_k=5)

In [109]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, toolkit.get_tools(), prompt=system_message)

C:\Users\User\AppData\Local\Temp\ipykernel_17820\1096870254.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, toolkit.get_tools(), prompt=system_message)


In [110]:
example_query = "How many products are there in the database?"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How many products are there in the database?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (d35219df-a933-4492-8520-b2fa84e18dd2)
 Call ID: d35219df-a933-4492-8520-b2fa84e18dd2
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

2017_budgets, customers, products, regions, sales_order, state_regions
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (490aa89d-b7e0-4c88-9fec-87d8311a270c)
 Call ID: 490aa89d-b7e0-4c88-9fec-87d8311a270c
  Args:
    table_names: products
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE products (
	`Id` INTEGER(11), 
	`Product Name` TEXT
)ENGINE=InnoDB COLLATE utf8mb4_general_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from products

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [ ]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompt for the API key if it isn't already set in the environment
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

try:
    # Initialize the model 
    # (Matches 'gemini-2.0-flash' from your previous code, but you can change this to 'gemini-1.5-flash' if needed)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    
    print("Sending test message to Gemini...")
    
    # Send a simple prompt to check the connection
    response = llm.invoke("Hello! This is a test message. Please reply exactly with: 'API Key is working!'.")
    
    print("\n✅ SUCCESS!")
    print(f"Gemini replied: {response.content}")

except Exception as e:
    print("\n❌ ERROR: Something went wrong with the API connection.")
    print(f"Details: {e}")


Sending test message to Gemini...

✅ SUCCESS!
Gemini replied: API Key is working!
